In [1]:
import pandas as pd
import numpy as np

# 1. Data load karo aur Timestamp ko datetime format me badlo
df = pd.read_csv('zepto_simulated_orders.csv')
df

FileNotFoundError: [Errno 2] No such file or directory: 'zepto_simulated_orders.csv'

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Timestamp']

In [ ]:
# 2. Timestamp se 'Hour' (ghanta) extract karo
df['Hour'] = df['Timestamp'].dt.hour


In [ ]:
# 3. Har hour par Status (Delivered vs Bounced) ka percentage nikalna
bounce_matrix = df.groupby('Hour')['Status'].value_counts(normalize=True).unstack().fillna(0)

In [ ]:
# 4. Bounce Rate % calculate karo
bounce_matrix['Bounce_Rate_%'] = bounce_matrix['Bounced (Out of Stock)'] * 100

In [ ]:
# Top 5 high-bounce hours dekho
# print("--- TOP 5 HIGH-BOUNCE HOURS ---")
print(bounce_matrix[['Bounce_Rate_%']].sort_values(by='Bounce_Rate_%', ascending=False))

In [ ]:
# 1. Har ghante kitne total clicks/orders aaye unhe count karo
hourly_demand = df.groupby('Hour').size().reset_index(name='Total_Orders')

# 2. Pandas/NumPy ka use karke 3-hour rolling average nikalna
hourly_demand['Rolling_Avg_3h'] = hourly_demand['Total_Orders'].rolling(window=3, min_periods=1).mean()

print("--- HOURLY DEMAND WITH 3-HOUR ROLLING AVERAGE ---")
print(hourly_demand.sort_values(by='Rolling_Avg_3h', ascending=False).head())

In [ ]:
# 1. Category wise Total Orders, Quantity aur Bounced Orders nikalna
category_analysis = df.groupby('Category').agg(
    Total_Orders=('Quantity_Ordered', 'count'),
    Total_Qty_Sold=('Quantity_Ordered', 'sum'),
    Bounced_Orders=('Status', lambda x: (x == 'Bounced (Out of Stock)').sum())
).reset_index()

# 2. Category wise Bounce Rate % nikalna
category_analysis['Category_Bounce_Rate_%'] = (category_analysis['Bounced_Orders'] / category_analysis['Total_Orders']) * 100

# Fast-Moving categories ke basis par sort karo
print("--- CATEGORY PERFORMANCE ---")
print(category_analysis.sort_values(by='Total_Qty_Sold', ascending=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting ke liye clean style set karte hain
sns.set_theme(style="whitegrid")

# -------------------------------------------------------------
# GRAPH 1: HOURLY DEMAND VS BOUNCE RATE (Dual Axis Graph)
# -------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(12, 6))

# Primary Axis: Total Orders (Bars)
color = 'tab:blue'
ax1.set_xlabel('Hour of the Day (0-23)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Orders (Demand)', color=color, fontsize=12, fontweight='bold')
ax1.bar(hourly_demand['Hour'], hourly_demand['Total_Orders'], color=color, alpha=0.5, label='Total Orders')
ax1.tick_params(axis='y', labelcolor=color)

# Secondary Axis: Bounce Rate % (Line)
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Customer Bounce Rate (%)', color=color, fontsize=12, fontweight='bold')
ax2.plot(bounce_matrix.index, bounce_matrix['Bounce_Rate_%'], color=color, linewidth=2.5, marker='o', label='Bounce Rate %')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Hourly Demand vs Customer Bounce Rate (Stock-Out Effect)', fontsize=15, fontweight='bold', pad=15)
fig.tight_layout()
plt.show() # Agar notebook me ho, to savefig bhi kar sakte ho: plt.savefig('hourly_bounce_demand.png', dpi=300)


# -------------------------------------------------------------
# GRAPH 2: CATEGORY-WISE BOUNCE RATE (Sorted Horizontal Bar)
# -------------------------------------------------------------
# Data ko sort kar lete hain taaki bar chart clean dikhe
category_analysis = category_analysis.sort_values(by='Category_Bounce_Rate_%', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x='Category_Bounce_Rate_%', y='Category', data=category_analysis, palette='coolwarm', ax=ax)

ax.set_xlabel('Customer Bounce Rate (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Product Category', fontsize=12, fontweight='bold')
ax.set_title('Customer Bounce Rate by Product Category', fontsize=15, fontweight='bold', pad=15)

plt.tight_layout()
plt.show() # plt.savefig('category_bounce_rate.png', dpi=300)